# Practice 50 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — project milestones

- Add `phase_end`: `kickoff` + `phase_months` months. Variable offset — use `.apply()`.
- Add `mid_review`: halfway through each phase, rounded to the nearest whole month.
- What day of the week does each `kickoff` fall on? Use `.dt.day_name()`.
- Flag `is_monday_kickoff`: True where `kickoff` is a Monday. `.dt.dayofweek` returns 0 for Monday.
- Sort the DataFrame by `phase_end` ascending.

In [15]:
milestones = pd.DataFrame({
    'project':      ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon'],
    'kickoff':      pd.to_datetime(['2024-01-08', '2024-03-01', '2024-05-15', '2024-07-22', '2024-10-07']),
    'phase_months': [3, 4, 2, 5, 6],
})

# Your code here
milestones['phase_end'] = milestones.apply(lambda row: row['kickoff'] + pd.DateOffset(months = row['phase_months']), axis = 1)
milestones['mid_review'] =  milestones.apply(lambda row: row['kickoff'] + pd.DateOffset(months = round(row['phase_months']/2)), axis = 1)

milestones['kickoff'].dt.day_name()
milestones['is_monday_kickoff'] = milestones['kickoff'].dt.dayofweek == 0


milestones = milestones.sort_values('phase_end', ascending=True)


In [16]:
milestones['kickoff'].dt.day_name()

0       Monday
1       Friday
2    Wednesday
3       Monday
4       Monday
Name: kickoff, dtype: object

In [13]:
milestones

,project,kickoff,phase_months,phase_end,mid_review,is_monday_kickoff
0,Alpha,2024-01-08,3,2024-04-08,2024-03-08,True
1,Beta,2024-03-01,4,2024-07-01,2024-05-01,False
2,Gamma,2024-05-15,2,2024-07-15,2024-06-15,False
3,Delta,2024-07-22,5,2024-12-22,2024-09-22,True
4,Epsilon,2024-10-07,6,2025-04-07,2025-01-07,True


---
## Level 2 — weekly sales

Use the weekly sales data below. No step-by-step — figure out the approach.

- Add `prev_week` and `week_4_ago`: sales from 1 and 4 weeks prior.
- Add `wow_pct`: week-over-week % change, rounded to 1 decimal place.
- Which week had the steepest drop? The steepest rise?
- Create `lagged`: a copy of the DataFrame with its index shifted forward by 4 weeks using `shift(freq=...)`. If you joined `sales` and `lagged` on the index, what would that let you compare?
- Add `above_4w_avg`: True where `units` exceeds the 4-week rolling mean. (Hint: `.rolling(4).mean()`)

In [34]:
sales = pd.DataFrame(
    {'units': [120, 135, 98, 150, 142, 168, 125, 180, 155, 172, 190, 160]},
    index=pd.date_range('2024-01-07', periods=12, freq='W')
)

# Your code here

sales['prev_week'] = sales['units'].shift(1)
sales['week_4_ago'] = sales['units'].shift(4)

sales['wow_pct'] = round((sales['units'] - sales['prev_week'])/sales['prev_week'],1)

sales['wow_pct'].idxmax()
sales['wow_pct'].idxmin()
lagged = sales.copy().shift(4, freq = 'W')

sales.join(lagged, how='inner', lsuffix="_current", rsuffix='_lagged')
### compare current with 4 weeks ago 

sales['rm'] = sales['units'].rolling(4).mean()
sales['above_4w_avg'] = sales['units']>sales['rm']

In [35]:
sales

,units,prev_week,week_4_ago,wow_pct,rm,above_4w_avg
2024-01-07,120,NaN,NaN,NaN,NaN,False
2024-01-14,135,120.0,NaN,0.1,NaN,False
2024-01-21,98,135.0,NaN,-0.3,NaN,False
2024-01-28,150,98.0,NaN,0.5,125.75,True
2024-02-04,142,150.0,120.0,-0.1,131.25,True
2024-02-11,168,142.0,135.0,0.2,139.50,True
2024-02-18,125,168.0,98.0,-0.3,146.25,False
2024-02-25,180,125.0,150.0,0.4,153.75,True
2024-03-03,155,180.0,142.0,-0.1,157.00,False
2024-03-10,172,155.0,168.0,0.1,158.00,True


---
## Level 3 — raw log parsing

Each row in `raw_logs` is an unparsed string. Use `.pipe()` to structure your work:

- **`parse(df)`** — use `.str.extract()` to pull `timestamp` (the datetime after `UTC:`), `system` (`AMER`, `EMEA`, or `APAC`), `level` (`INFO` or `ERROR`), and `bytes_transferred` (integer) from the `raw` column.
- **`add_timezones(df)`** — localize `timestamp` to UTC. Add `local_hour` by converting each row to its system's timezone with `.apply()` and extracting `.hour`:
  - `AMER` → `US/Eastern`
  - `EMEA` → `Europe/London`
  - `APAC` → `Asia/Tokyo`
- **`classify(df)`** — convert `system` and `level` to `category`. Add `is_error`: True where `level == 'ERROR'`. Add `transfer_tier`: `'low'` (< 1000 bytes), `'high'` (≥ 1000).

After the chain:
- Mean `bytes_transferred` per `system`.
- Among errors, what is the most common `local_hour`?
- What fraction of `high` transfer events are errors?

In [64]:
raw_logs = pd.DataFrame({'raw': [
    'UTC:2024-03-15T09:00 | SYS:AMER | level=INFO  | bytes=2048',
    'UTC:2024-03-15T14:30 | SYS:EMEA | level=ERROR | bytes=512',
    'UTC:2024-03-15T01:00 | SYS:APAC | level=INFO  | bytes=4096',
    'UTC:2024-03-15T18:00 | SYS:AMER | level=ERROR | bytes=256',
    'UTC:2024-03-15T07:00 | SYS:EMEA | level=INFO  | bytes=1024',
    'UTC:2024-03-15T22:00 | SYS:APAC | level=ERROR | bytes=768',
    'UTC:2024-03-16T11:00 | SYS:AMER | level=INFO  | bytes=3072',
    'UTC:2024-03-16T04:00 | SYS:EMEA | level=ERROR | bytes=128',
    'UTC:2024-03-16T15:00 | SYS:APAC | level=INFO  | bytes=512',
    'UTC:2024-03-16T20:00 | SYS:AMER | level=ERROR | bytes=1536',
]})

# Your code here
def parse(df):
    df['timestamp'] = pd.to_datetime(df['raw'].str.extract(r'UTC:(\S+)')[0].str.replace('T',' '))
    df['system'] = df['raw'].str.extract(r'SYS:(\w+)')[0]
    df['level'] = df['raw'].str.extract(r'level=(\w+)')[0]
    df['bytes_transferred'] = df['raw'].str.extract(r'bytes=(\d+)')[0].astype(int)
    return df 

def add_timezones(df):
    df['timestamp'] = df['timestamp'].dt.tz_localize('UTC')
    df['local_hour'] = df.apply(lambda row: row['timestamp'].tz_convert('US/Eastern').hour if row['system'] == 'AMER'
                                else row['timestamp'].tz_convert('Europe/London').hour if row['system'] == 'EMEA'
                                else row['timestamp'].tz_convert('Asia/Tokyo').hour, axis = 1)
    return df 

def classify(df):
    df[['system','level']] = df[['system','level']].astype('category')
    df['is_error'] = df['level'] == 'ERROR'
    df['transfer_tier'] = pd.cut(df['bytes_transferred'], bins = [0,1000,np.inf], labels=['low','high'], right=False)
    return df


After the chain:
- Mean `bytes_transferred` per `system`.
- Among errors, what is the most common `local_hour`?
- What fraction of `high` transfer events are errors?

In [59]:
res = raw_logs.copy().pipe(parse).pipe(add_timezones).pipe(classify)


In [63]:
res.groupby('system', observed=True)['bytes_transferred'].mean()
res.loc[res['is_error']]['local_hour'].value_counts().idxmax()
res.loc[res['transfer_tier']=='high','is_error'].mean()

np.float64(0.2)